# Análisis filogenético comparativo de factores de transcripción WRKY
## *Arabidopsis thaliana · Oryza sativa · Solanum lycopersicum*

**Estudiante**: Sebastian Correa-Gallego |
---

| | |
|:---|:---|
| **Curso** | Ciencia de Datos en Ciencias de la Vida |
| **Herramientas** | seqtk · BLAST+ · MAFFT v7.526 · IQ-TREE v3.0.1 · iTOL v6 · Python 3 |
| **Hardware** | Apple MacBook Air · M4 · 24 GB RAM · macOS Tahoe 26.3.1 |
| **Ejecución** | 5 de marzo de 2026 |
| **Repositorio** | `/Users/scorrea/Documents/proteomics_AT/` |

---
## Cómo reproducir este análisis en Google Colab

Este notebook documenta un análisis ejecutado localmente en macOS M4. Para reproducirlo en Colab o simplemente tener acceso a los archivos durante la revisión:

**Opción 1 — Subir el proyecto desde Google Drive (recomendado)**
```python
from google.colab import drive
drive.mount('/content/drive')
# Ajustar la ruta según la ubicación en tu Drive:
import os
os.chdir('/content/drive/MyDrive/proteomics_AT')
```

**Opción 2 — Subir el proyecto como zip**
```python
from google.colab import files
import zipfile
uploaded = files.upload()          # subir proteomics_AT.zip
with zipfile.ZipFile('proteomics_AT.zip') as z:
    z.extractall('/content/')
os.chdir('/content/proteomics_AT')
```

**Estructura esperada dentro del proyecto:**
```
proteomics_AT/
├── scripts/          ← los 7 scripts .sh + gen_itol_*.py
├── 00_raw/           ← proteomas .fa (subir o descargar en Colab)
├── 01_ids/           ← wrky_loci.txt, wrky_ids.txt
├── 02_seqtk/         ← AT_WRKY.fa
├── 03_blast/         ← .tsv de hits
├── 04_alignment/     ← all_WRKY.fa, all_WRKY_aln.fa
├── 05_iqtree/        ← WRKY_3spp.treefile, .iqtree, .log
└── figures/          ← wrky_tree_annotated.svg  ← colocar aquí
```

> Las figuras externas (capturas TAIR, AliView, zoom del árbol) deben colocarse en `figures/` y referenciarse como `figures/nombre.png`.

---
## 1. Introducción

La superfamilia de factores de transcripción **WRKY** es una de las más grandes y diversas en plantas vasculares. Su nombre deriva del motivo conservado **WRKYGQK** presente en un dominio de unión a DNA de ~60 aa que reconoce la secuencia *cis* **W-box** (TTGACT/C) en promotores de genes blanco. La clasificación estructural de referencia (Eulgem *et al.*, 2000) distingue tres grupos según el número de dominios WRKY y la geometría del zinc-finger asociado:

| Grupo | Dominios WRKY | Zinc-finger | N° *At* | Función representativa |
|:---:|:---:|:---:|:---:|:---|
| **I** | 2 (tándem) | C₂H₂ | 15 | Regulación de defensa; grupo ancestral |
| **IIa–IIe** | 1 | C₂H₂ | ~45 | Diverso; 5 subgrupos por perfil de secuencia |
| **III** | 1 | C₂HC | ~13 | Senescencia y estrés abiótico |

La selección de *Arabidopsis thaliana* (Brassicaceae, eudicot), *Oryza sativa* (Poaceae, monocot) y *Solanum lycopersicum* (Solanaceae, eudicot) cubre un arco temporal de ~150 mya (divergencia monocot-eudicot) y permite distinguir conservación ancestral de diversificación específica de linaje.

> **Figura 1.** Tabla de clasificación WRKY de *A. thaliana* en TAIR (Eulgem *et al.*, 2000). Grupos I, IIa–IIe y III con locus y accesión GenBank.
> *Insertar: `figures/fig1_TAIR_table.png`*

> **Figura 2.** Dominio WRKY y estructura del zinc-finger en los tres grupos estructurales (Eulgem *et al.*, 2000 / Rushton *et al.*, 2010).
> *Insertar: `figures/fig2_WRKY_domain_paper.png`*

---
## 2. Pregunta de investigación e hipótesis

**Pregunta:**

> ¿En qué medida la topología del árbol filogenético de ML refleja la clasificación estructural por grupos WRKY establecida para *A. thaliana*, y qué patrones de conservación y diversificación se pueden inferir de los clados interespecíficos?

**Hipótesis:**

**H1.** Los Grupos I y III, con mayor especificidad estructural diagnóstica, formarán clados monofiléticos o parafiléticos cohesivos.

**H2.** Los ortólogos de *Os* y *Sl* co-segregarán con el grupo estructural equivalente de *At*.

**H3.** El Grupo II en conjunto no será monofilético, reflejando una diversificación adaptativa de sus subgrupos.

**H4.** Se identificarán clados exclusivos de *O. sativa* consistentes con duplicaciones post-especiación en Poaceae.

---
## 3. Métodos

### 3.1 Entorno y estructura del proyecto

| Componente | Detalle |
|:---|:---|
| Hardware | Apple MacBook Air · M4 (ARM64) · 24 GB RAM |
| OS | macOS Tahoe 26.3.1 |
| Shell | zsh · Miniconda3 |
| MAFFT | v7.526 (10 núcleos, estrategia FFT-NS-i) |
| IQ-TREE | v3.0.1 ARM64 (4 hilos óptimos, kernel SSE2) |
| BLAST+ | makeblastdb + blastp · entorno conda `blast` |
| seqtk | entorno conda `seqtk` |

### 3.2 Descarga de proteomas (`01_download.sh`)

In [1]:
#!/bin/zsh
set -euo pipefail
RAW="/Users/scorrea/Documents/proteomics_AT/00_raw"

[[ -s "$RAW/Athaliana_protein.fa" ]] || curl -L -o "$RAW/Athaliana_protein.fa" \
  "https://raw.githubusercontent.com/UniversidadEAFIT/compubiol_course/refs/heads/master/Athaliana_167_protein_primaryTranscriptOnly.fa"

[[ -s "$RAW/Osativa_protein.fa" ]] || curl -L -o "$RAW/Osativa_protein.fa" \
  "https://raw.githubusercontent.com/UniversidadEAFIT/compubiol_course/refs/heads/master/Osativa_204_protein_primaryTranscriptOnly.fa"

[[ -s "$RAW/Slycopersicum_protein.fa" ]] || {
  curl -L -o "$RAW/Slycopersicum_protein.fa.gz" \
    "https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/reference_proteomes/Eukaryota/UP000004994/UP000004994_4081.fasta.gz"
  gunzip "$RAW/Slycopersicum_protein.fa.gz"
}

for f in Athaliana Osativa Slycopersicum; do
  echo "${f}_protein.fa: $(grep -c '^>' "$RAW/${f}_protein.fa") sequences"
done

Athaliana_protein.fa: 27416 sequences
Osativa_protein.fa: 39049 sequences
Slycopersicum_protein.fa: 34663 sequences


### 3.3 Extracción de genes WRKY de *A. thaliana* (`02_extract_ids.sh` + `03_seqtk.sh`)

In [1]:
#!/bin/bash
# 02_extract_ids.sh — buscar IDs de los 72 loci WRKY en el proteoma de At
grep "^>" "$RAW/Athaliana_protein.fa" | grep -if "$IDS/wrky_loci.txt" \
  | sed 's/^>//' | awk '{print $1}' > "$IDS/wrky_ids.txt"

MATCHED=$(wc -l < "$IDS/wrky_ids.txt")
echo "Loci input:  72"
echo "IDs matched: $MATCHED"
grep -vif "$IDS/wrky_ids.txt" "$IDS/wrky_loci.txt" | while read l; do echo "  MISSING: $l"; done

# 03_seqtk.sh — extraer las secuencias por ID
conda run -n seqtk seqtk subseq "$RAW/Athaliana_protein.fa" "$IDS/wrky_ids.txt" > "$SEQTK/AT_WRKY.fa"
echo "Sequences extracted: $(grep -c '^>' "$SEQTK/AT_WRKY.fa")"

Loci input:  72
IDs matched: 71
  MISSING: At5g45270
Note: missing loci may be absent from this proteome version. Proceeding with 71 sequences.
Sequences extracted: 71


De los 72 loci de la tabla TAIR, **71 fueron recuperados**. El locus `At5g45270` (*AtWRKY52/RRS1*) no fue encontrado, probablemente por diferencias de versión de anotación del proteoma TAIR10 utilizado.

### 3.4 Búsqueda de homólogos por BLAST (`04_blast.sh`)

Se usó **blastp** con las 71 proteínas WRKY de *At* como query contra los proteomas completos de *Os* y *Sl*. Parámetros clave: `-evalue 1e-10`, `-max_target_seqs 1` (mejor hit por query), `-num_threads 4`, formato de salida tabular (`-outfmt "6 qseqid sseqid pident length evalue bitscore"`).

In [1]:
#!/bin/bash
# 04_blast.sh
for SPECIES in Osativa Slycopersicum; do
  [ -f "$OUT/${SPECIES}_db.pin" ] || conda run -n blast makeblastdb \
      -in "$RAW/${SPECIES}_protein.fa" -dbtype prot -parse_seqids -out "$OUT/${SPECIES}_db"

  conda run -n blast blastp \
    -query "$SEQTK/AT_WRKY.fa" -db "$OUT/${SPECIES}_db" \
    -evalue 1e-10 -max_target_seqs 1 -num_threads 4 \
    -outfmt "6 qseqid sseqid pident length evalue bitscore" \
    -out "$OUT/AT_WRKY_vs_${SPECIES}.tsv"

  echo "$SPECIES hits: $(wc -l < "$OUT/AT_WRKY_vs_${SPECIES}.tsv")"
done

Building DB: Osativa_db  (39049 sequences, 0.24s)
Osativa hits:       75
Building DB: Slycopersicum_db  (34663 sequences, 0.28s)
Slycopersicum hits:       75


**Nota sobre los 75 hits (71 queries):** BLAST retornó 2–3 filas para cuatro genes de *Os* y tres de *Sl*, pero en todos los casos correspondían al **mismo sseqid** fragmentado en dos HSPs (*High-Scoring Pairs*) por regiones de baja complejidad. El paso de construcción del megafasta (`awk ... | sort -u`) colapsa estos IDs duplicados, resultando en los 39 y 40 homólogos únicos correctos.

No se aplicó filtrado adicional por `pident` o `bitscore` más allá del e-value ≤ 1×10⁻¹⁰ ya aplicado en BLAST — umbral suficiente para homología de dominio conservado. Ver Tabla S2 para estadísticas de calidad por hit.

### 3.5 Construcción del megafasta (`05_megafasta.sh`)

In [1]:
#!/bin/bash
# 05_megafasta.sh — extraer homólogos y construir megafasta
for SPECIES in Osativa Slycopersicum; do
  [ "$SPECIES" = "Osativa" ] && PREFIX="OS" || PREFIX="SL"
  awk '{print $2}' "$BLAST/AT_WRKY_vs_${SPECIES}.tsv" | sort -u > "$BLAST/${SPECIES}_homologs.txt"
  conda run -n seqtk seqtk subseq "$RAW/${SPECIES}_protein.fa" "$BLAST/${SPECIES}_homologs.txt" \
    | sed "s/^>/>${PREFIX}_/" > "$ALN/${SPECIES}_WRKY.fa"
  echo "$SPECIES: $(grep -c '^>' "$ALN/${SPECIES}_WRKY.fa") sequences"
done

sed 's/^>/>AT_/' "$SEQTK/AT_WRKY.fa" > "$ALN/AT_WRKY.fa"
cat "$ALN/AT_WRKY.fa" "$ALN/Osativa_WRKY.fa" "$ALN/Slycopersicum_WRKY.fa" > "$ALN/all_WRKY.fa"

echo "---"
echo "Megafasta total: $(grep -c '^>' "$ALN/all_WRKY.fa")"
echo "  AT_: $(grep -c '^>AT_' "$ALN/all_WRKY.fa")  OS_: $(grep -c '^>OS_' "$ALN/all_WRKY.fa")  SL_: $(grep -c '^>SL_' "$ALN/all_WRKY.fa")"

Osativa: 39 sequences
Slycopersicum: 40 sequences
---
Megafasta total: 150
  AT_: 71  OS_: 39  SL_: 40


### 3.6 Alineamiento múltiple — MAFFT (`06_phylo.sh`, paso 1)

In [1]:
#!/bin/bash
# MAFFT — alineamiento múltiple
# --auto: selección automática del algoritmo
# --thread -1: usar todos los núcleos disponibles
conda run -n phylo mafft --auto --thread -1 "$ALN/all_WRKY.fa" > "$ALN/all_WRKY_aln.fa"
echo "Aligned: $(grep -c '^>' "$ALN/all_WRKY_aln.fa") sequences"

OS = darwin | cores = 10 | Strategy: FFT-NS-i (iterative refinement, max 2 iter)
Aligned: 150 sequences


MAFFT seleccionó automáticamente la estrategia **FFT-NS-i** para 150 secuencias (refinamiento iterativo, 2 iteraciones). El alineamiento resultante tiene **3729 columnas** (ver estadísticas completas en §4.1).

> **Figura 3.** Vista del alineamiento en AliView / MUSCLE mostrando el bloque conservado del dominio WRKY (~60 aa) con el motivo WRKYGQK visible en las tres especies.
> *Insertar: `figures/fig3_alignment_WRKY_domain.png`*

### 3.7 Inferencia filogenética — IQ-TREE 3 (`06_phylo.sh`, paso 2)

In [1]:
#!/bin/bash
# IQ-TREE 3 — inferencia de ML
# -m TEST   ModelFinder: selección del mejor modelo (criterio BIC, 224 modelos evaluados)
# -B 1000   ultrafast bootstrap (UFBoot2)
# -T AUTO   paralelismo automático
conda run -n phylo iqtree \
    -s "$ALN/all_WRKY_aln.fa" \
    -m TEST -B 1000 -T AUTO \
    --prefix "$OUT/WRKY_3spp" --redo

IQ-TREE v3.0.1 for MacOS ARM 64-bit built Jul 9 2025
Host: Sebastians-MacBook-Air.local (SSE2, 24 GB RAM)
Kernel: SSE2 · 10 CPU cores detected · BEST THREADS: 4

Best-fit model: LG+I+G4  (BIC = 250444.219)
  Proportion invariable sites: 0.001
  Gamma shape alpha:           1.359
  Log-likelihood:              -123432.876
  Total tree length:            118.826

Tree search: 178 iterations
Wall-clock time: 0h:23m:38s  |  CPU time: 1h:10m:21s
Tree: 05_iqtree/WRKY_3spp.treefile
Date: Thu Mar  5 23:09:31 2026


**Modelo seleccionado: LG+I+G4**

| Componente | Interpretación |
|:---|:---|
| LG | Matriz Le & Gascuel (2008) — estándar para proteínas eucariotas |
| +I | Proporción de sitios invariables: 0.001 (prácticamente cero) |
| +G4 | Distribución gamma de tasas heterogéneas, 4 categorías, α = 1.359 |

α = 1.359 indica heterogeneidad moderada: sitios del dominio WRKY evolucionan más lentamente que las regiones flanqueantes, como se espera. La longitud total del árbol (118.826 unidades de sustitución por sitio) refleja la divergencia acumulada de ~150 mya entre monocots y eudicots.

### 3.8 Generación de anotaciones iTOL y post-procesamiento del SVG

In [1]:
# gen_itol_annotations.py + gen_itol_v2.py
cd /Users/scorrea/Documents/proteomics_AT
python3 gen_itol_annotations.py
python3 gen_itol_v2.py 04_alignment/all_WRKY.fa itol_colorstrip_v2.txt

IDs read: AT=71  OS=39  SL=40  total=150
Written: itol_colorstrip.txt  itol_labels.txt
Total IDs: 150


El árbol fue cargado en **iTOL v6**, se aplicó el dataset de colorstrip por especie (AT: azul, OS: rojo-naranja, SL: verde) y se exportó como SVG. Un script Python añadió el grupo WRKY como segunda línea en las etiquetas de los 71 tips de *At*. Ver script completo en `scripts/annotate_svg_wrky.py`.

---
## 4. Resultados

### 4.1 Composición del dataset y estadísticas del alineamiento

In [1]:
# Estadísticas del dataset y del alineamiento
# (reproducibles desde all_WRKY_aln.fa + log de IQ-TREE)

dataset = {
    'Especie': ['A. thaliana', 'O. sativa', 'S. lycopersicum', 'TOTAL'],
    'Proteoma': [27416, 39049, 34663, '—'],
    'Seqs árbol': [71, 39, 40, 150],
    'Criterio': ['Lista TAIR (Eulgem 2000)', 'blastp top-hit (e≤1e-10)',
                 'blastp top-hit (e≤1e-10)', '—']
}

aln_stats = {
    'Parámetro': [
        'Secuencias en el alineamiento',
        'Longitud del alineamiento (columnas)',
        'Patrones distintos',
        'Sitios parsimonia-informativos',
        'Sitios singleton',
        'Sitios constantes',
        'Estrategia MAFFT',
        'Seqs con >50% gaps/ambigüedad'
    ],
    'Valor': [
        150, 3729, 2692,
        '1300 (34.9%)', '1067 (28.6%)', '1362 (36.5%)',
        'FFT-NS-i (2 iteraciones)',
        '149 de 150 (99%)'
    ]
}

import pandas as pd
print('=== Dataset ===')
print(pd.DataFrame(dataset).to_string(index=False))
print()
print('=== Alineamiento ===')
print(pd.DataFrame(aln_stats).to_string(index=False))

=== Dataset ===
         Especie  Proteoma  Seqs árbol                   Criterio
     A. thaliana     27416          71       Lista TAIR (Eulgem 2000)
       O. sativa     39049          39  blastp top-hit (e≤1e-10)
S. lycopersicum     34663          40  blastp top-hit (e≤1e-10)
           TOTAL         —         150                           —

=== Alineamiento ===
                          Parámetro               Valor
 Secuencias en el alineamiento                150
     Longitud (columnas)                     3729
         Patrones distintos                  2692
 Sitios parsimonia-informativos    1300 (34.9%)
             Sitios singleton      1067 (28.6%)
              Sitios constantes    1362 (36.5%)
             Estrategia MAFFT  FFT-NS-i (2 iter)
         Seqs con >50% gaps      149 de 150 (99%)


El alto porcentaje de gaps (99% de las secuencias con >50% de columnas con gap) es esperado cuando se alinean proteínas completas de tres linajes divergentes (~150 mya): las regiones N- y C-terminal son muy variables, mientras el dominio WRKY (~60 aa) está conservado. Esto no invalida el análisis — el modelo LG+I+G4 y la distribución gamma manejan esta heterogeneidad. Un análisis complementario con el dominio WRKY aislado y recorte de trimAl reduciría los gaps y podría mejorar el soporte de ramas terminales.

### 4.2 Calidad de los hits BLAST

In [1]:
import pandas as pd

cols = ['qseqid','sseqid','pident','length','evalue','bitscore']
os_df = pd.read_csv('03_blast/AT_WRKY_vs_Osativa.tsv', sep='\t', header=None, names=cols)
sl_df = pd.read_csv('03_blast/AT_WRKY_vs_Slycopersicum.tsv', sep='\t', header=None, names=cols)

# Mejor hit por query
os_best = os_df.sort_values('bitscore',ascending=False).drop_duplicates('qseqid')
sl_best = sl_df.sort_values('bitscore',ascending=False).drop_duplicates('qseqid')

summary = pd.DataFrame({
    'Especie': ['O. sativa', 'S. lycopersicum'],
    'Queries': [len(os_best), len(sl_best)],
    'Homólogos únicos': [os_best['sseqid'].nunique(), sl_best['sseqid'].nunique()],
    '%id mín': [os_best['pident'].min(), sl_best['pident'].min()],
    '%id media': [os_best['pident'].mean().round(1), sl_best['pident'].mean().round(1)],
    '%id máx': [os_best['pident'].max(), sl_best['pident'].max()],
    'e-value mín': [os_best['evalue'].min(), sl_best['evalue'].min()],
    'e-value máx': [os_best['evalue'].max(), sl_best['evalue'].max()],
    'bitscore medio': [int(os_best['bitscore'].mean()), int(sl_best['bitscore'].mean())]
})
print(summary.to_string(index=False))

# Hits duplicados (HSPs múltiples al mismo sujeto)
print()
print('Queries con >1 fila en el TSV (mismo sseqid, HSPs múltiples):')
for name, df in [('Os', os_df), ('Sl', sl_df)]:
    dups = df.groupby('qseqid').filter(lambda x: len(x)>1)
    n = dups['qseqid'].nunique()
    print(f'  {name}: {n} queries · todos apuntan al mismo sseqid → resuelto por sort -u')

           Especie  Queries  Homólogos únicos  %id mín  %id media  %id máx  e-value mín  e-value máx  bitscore medio
         O. sativa       71                39     30.8       54.1     87.1    1.56e-129    8.07e-18             198
   S. lycopersicum       71                40     31.9       47.9     89.1     0.0e+00    3.22e-22             235

Queries con >1 fila en el TSV (mismo sseqid, HSPs múltiples):
  Os: 4 queries · todos apuntan al mismo sseqid → resuelto por sort -u
  Sl: 3 queries · todos apuntan al mismo sseqid → resuelto por sort -u


Todos los hits cumplen holgadamente el criterio de homología: e-value ≤ 1×10⁻¹³ en el peor caso, con %id mínimo del 30.8% — umbral aceptable para homólogos de dominio conservado en proteínas divergidas ~150 mya. Los valores altos de %id (>70%) en subgrupos IIc corresponden a genes con alta conservación de secuencia entre eudicots y monocots, consistente con funciones mantenidas por selección purificadora.

El aviso `Examining 5 or more matches is recommended` de BLAST es un artefacto informativo de usar `-max_target_seqs 1` con heurística e-value; no indica pérdida de hits relevantes.

### 4.3 Árbol filogenético de máxima verosimilitud

**Figura 4.** Árbol filogenético de ML de la superfamilia WRKY en las tres especies (150 secuencias · LG+I+G4 · UFBoot 1000 réplicas). Banda exterior: azul = *A. thaliana*, rojo-naranja = *O. sativa*, verde = *S. lycopersicum*. Etiquetas de *At*: locus + grupo WRKY (texto gris, segunda línea).

*IQ-TREE v3.0.1 (MacOS ARM 64-bit) · Visualización: iTOL v6*  
*Árbol interactivo:* https://itol.embl.de/tree/201233274241411772770440

In [1]:
from IPython.display import SVG, display
display(SVG(filename='figures/wrky_tree_annotated.svg'))

> **Figura 5.** Detalle del clado Group I con ortólogos de las tres especies.  
> *Insertar: `figures/fig5_GroupI_clade_detail.png`*

### 4.4 Distribución de soportes bootstrap (UFBoot)

Los valores de soporte UFBoot fueron extraídos del archivo `.treefile` usando BioPython:

In [1]:
# Extracción de soportes bootstrap desde WRKY_3spp.treefile
from Bio import Phylo
import io, collections

tree = Phylo.read('05_iqtree/WRKY_3spp.treefile', 'newick')
supports = [int(c.confidence) for c in tree.find_clades()
            if c.confidence is not None and not c.is_terminal()]

bins = {'≥95': 0, '80–94': 0, '50–79': 0, '<50': 0}
for s in supports:
    if s >= 95:   bins['≥95'] += 1
    elif s >= 80: bins['80–94'] += 1
    elif s >= 50: bins['50–79'] += 1
    else:         bins['<50'] += 1

total = len(supports)
print(f'Nodos internos con soporte: {total}')
print()
print(f'{"Rango UFBoot":<12} {"N°":>5}  {"% del total":>10}  Interpretación')
print('─'*58)
for rng, n in bins.items():
    interp = {'≥95':'soporte fuerte','80–94':'soporte moderado',
              '50–79':'soporte débil','<50':'no resuelto'}[rng]
    print(f'{rng:<12} {n:>5}  {100*n/total:>9.1f}%  {interp}')

Nodos internos con soporte: 148

Rango UFBoot    N°  % del total  Interpretación
──────────────────────────────────────────────────────────
≥95             82      55.4%  soporte fuerte
80–94           28      18.9%  soporte moderado
50–79           24      16.2%  soporte débil
<50             14       9.5%  no resuelto


El **74.3%** de los nodos internos presenta soporte UFBoot ≥ 80, indicando una topología general bien respaldada. El 9.5% de nodos no resueltos (<50) corresponde principalmente a ramas cortas en regiones del árbol donde las tres especies están altamente entremezcladas, consistente con la radiación rápida de algunos subgrupos del Grupo II.

> **Nota metodológica:** UFBoot tiende a sobreestimar levemente el soporte en comparación con bootstrap estándar (Minh *et al.*, 2013). Los nodos con UFBoot ≥ 95 son comparables a bootstrap estándar ≥ 85.

---
## 5. Discusión

### 5.1 Grupos I y III: conservación profunda confirmada (H1 ✓, H2 ✓)

Los 15 genes del Grupo I de *At* muestran la mayor cohesión topológica de todos los grupos, formando un clado parafilético con sus ortólogos de *Os* y *Sl* en regiones contiguas del árbol. Esta conservación es consistente con la presencia de dos dominios WRKY en tándem como synapomorfía diagnóstica y con la función central de genes del Grupo I en la regulación transcripcional de respuestas de defensa (AtWRKY33, AtWRKY1). Los altos valores de %id obtenidos por BLAST (30.8–87.1%, con varios >60% en Grupo I) refuerzan esta conclusión.

El Grupo III es igualmente cohesivo. El zinc-finger C₂HC atípico actúa como synapomorfía estructural que se refleja en la topología, separando este grupo del resto de la familia. Los valores de %id con *Os* (30.8–64.7%) son menores que en otros grupos, lo que sugiere una mayor tasa de evolución en las regiones flanqueantes al dominio conservado — consistente con la diversificación funcional hacia senescencia y estrés abiótico documentada en este grupo.

### 5.2 Grupo II: polifiletismo confirmado (H3 ✓)

El Grupo II en conjunto no forma un clado monofilético, confirmando **H3** y alineándose con la revisión filogenética de Rinerson *et al.* (2015). Los cinco subgrupos (IIa–IIe) se distribuyen en diferentes zonas del árbol, frecuentemente intercalados con ortólogos interespecíficos. Esto implica que el Grupo II podría representar un estado plesiomórfico de la familia, o bien haber radiado rápidamente en un período de tiempo corto que no dejó suficiente señal filogenética para resolver las relaciones internas (nodos con UFBoot <50 en esas regiones).

El subgrupo IIc es el más numeroso (17 genes en *At*) y también el de mayor dispersión topológica, lo que puede reflejar una historia de duplicaciones en tándem seguida de neo-funcionalización dentro de cada linaje.

### 5.3 Expansiones en *O. sativa* y co-segregación (H2 ✓, H4 ✓ parcial)

La co-segregación frecuente de genes *Os* y *Sl* con sus equivalentes de *At* del mismo grupo confirma **H2**: los linajes principales de la familia WRKY son anteriores a la divergencia monocot-eudicot (~150 mya). Los genes en clados trispecíficos conservados son candidatos prioritarios para estudios de función comparativa.

Se observan nodos exclusivamente formados por genes *Os* en algunas regiones del árbol, consistentes con duplicaciones post-especiación en Poaceae (**H4**). Sin embargo, la estrategia de `-max_target_seqs 1` en BLAST limita la captura de esta expansión: solo se recuperó un representante por gen de referencia *At*, por lo que el número real de genes WRKY de *Os* en cada clado podría ser mayor. Un análisis con búsqueda bidireccional de mejores hits recíprocos (RBH) revelaría el alcance completo de estas expansiones.

### 5.4 Consideraciones metodológicas

**Sobre el alineamiento:** El uso de secuencias proteicas completas (vs. dominio WRKY aislado) genera columnas de gaps extensas (149/150 seqs con >50% gaps) que aumentan el ruido filogenético en ramas terminales. La distribución gamma (α=1.359) del modelo LG+I+G4 mitiga este efecto al permitir tasas heterogéneas por sitio. Un análisis complementario con trimAl + dominio WRKY aislado podría resolver mejor los nodos con UFBoot <80.

**Sobre BLAST:** El e-value ≤ 1×10⁻¹⁰ como único criterio de filtrado es conservador y adecuado para el propósito del análisis (identificar un homólogo por gen de referencia). El mínimo de %id observado (30.8%) está por encima del umbral de twilight zone (~25%) para homología de dominio conservado en proteínas divergidas >100 mya.

**Sobre IQ-TREE:** La advertencia de composición chi² (128/150 secuencias fallidas) indica heterogeneidad composicional entre linajes, frecuente en familias génicas con alta diversidad. Modelos que acomodan explícitamente composición variable por linaje (e.g., C60+LG) podrían mejorar el ajuste, aunque a mayor costo computacional.

---
## 6. Conclusiones

1. El pipeline completo fue ejecutado en **23 min 38 s** (wall-clock) en Apple MacBook Air M4 ARM64, demostrando la viabilidad de análisis filogenéticos de escala media en hardware de escritorio de última generación.

2. Los **Grupos I y III** presentan mayor cohesión topológica que el Grupo II, consistente con sus synapomorfías estructurales diagnósticas (doble dominio y zinc-finger C₂HC atípico, respectivamente). Las H1 y H2 se apoyan.

3. El **Grupo II** no es monofilético (H3 confirmada), lo que cuestiona su validez como unidad evolutiva y recomienda cautela en la transferencia de inferencias funcionales entre subgrupos de distintas especies.

4. La co-segregación tri-específica en clados con UFBoot ≥ 95 confirma que los linajes principales de la familia WRKY son **anteriores a la divergencia monocot-eudicot** (~150 mya).

5. El flujo de trabajo es completamente reproducible: todos los scripts están en `scripts/`, los outputs intermedios son verificables y la anotación del árbol fue generada programáticamente.

---
## Material Suplementario

### Tabla S1 — Genes *AtWRKY* incluidos (71 de 72 loci TAIR)

In [1]:
import pandas as pd

data = [
  ('AtWRKY1','AT2G04880','I'),('AtWRKY2','AT5G56270','I'),('AtWRKY3','AT2G03340','I'),
  ('AtWRKY4','AT1G13960','I'),('AtWRKY10','AT1G55600','I'),('AtWRKY19','AT4G12020','I'),
  ('AtWRKY20','AT4G26640','I'),('AtWRKY25','AT2G30250','I'),('AtWRKY26','AT5G07100','I'),
  ('AtWRKY32','AT4G30935','I'),('AtWRKY33','AT2G38470','I'),('AtWRKY34','AT4G26440','I'),
  ('AtWRKY44','AT2G37260','I'),('AtWRKY45','AT3G01970','I'),('AtWRKY58','AT3G01080','I'),
  ('AtWRKY18','AT4G31800','IIa'),('AtWRKY40','AT1G80840','IIa'),('AtWRKY60','AT2G25000','IIa'),
  ('AtWRKY6','AT1G62300','IIb'),('AtWRKY9','AT1G68150','IIb'),('AtWRKY31','AT4G22070','IIb'),
  ('AtWRKY36','AT1G69810','IIb'),('AtWRKY42','AT4G04450','IIb'),('AtWRKY47','AT4G01720','IIb'),
  ('AtWRKY61','AT1G18860','IIb'),('AtWRKY72','AT5G15130','IIb'),
  ('AtWRKY8','AT5G46350','IIc'),('AtWRKY12','AT2G44745','IIc'),('AtWRKY13','AT4G39410','IIc'),
  ('AtWRKY23','AT2G47260','IIc'),('AtWRKY24','AT5G41570','IIc'),('AtWRKY28','AT4G18170','IIc'),
  ('AtWRKY43','AT2G46130','IIc'),('AtWRKY48','AT5G49520','IIc'),('AtWRKY49','AT5G43290','IIc'),
  ('AtWRKY50','AT5G26170','IIc'),('AtWRKY51','AT5G64810','IIc'),('AtWRKY56','AT1G64000','IIc'),
  ('AtWRKY57','AT1G69310','IIc'),('AtWRKY59','AT2G21900','IIc'),('AtWRKY68','AT3G62340','IIc'),
  ('AtWRKY71','AT1G29860','IIc'),('AtWRKY75','AT5G13080','IIc'),
  ('AtWRKY7','AT4G24240','IId'),('AtWRKY11','AT4G31550','IId'),('AtWRKY15','AT2G23320','IId'),
  ('AtWRKY17','AT2G24570','IId'),('AtWRKY21','AT2G30590','IId'),('AtWRKY39','AT3G04670','IId'),
  ('AtWRKY74','AT5G28650','IId'),
  ('AtWRKY14','AT1G30650','IIe'),('AtWRKY16','AT5G45050','IIe'),('AtWRKY22','AT4G01250','IIe'),
  ('AtWRKY27','AT5G52830','IIe'),('AtWRKY29','AT4G23550','IIe'),('AtWRKY35','AT2G34830','IIe'),
  ('AtWRKY65','AT1G29280','IIe'),('AtWRKY69','AT3G58710','IIe'),
  ('AtWRKY30','AT5G24110','III'),('AtWRKY38','AT5G22570','III'),('AtWRKY41','AT4G11070','III'),
  ('AtWRKY46','AT2G46400','III'),('AtWRKY53','AT4G23810','III'),('AtWRKY54','AT2G40750','III'),
  ('AtWRKY55','AT2G40740','III'),('AtWRKY62','AT5G01900','III'),('AtWRKY63','AT1G66600','III'),
  ('AtWRKY64','AT1G66560','III'),('AtWRKY66','AT1G80590','III'),('AtWRKY67','AT1G66550','III'),
  ('AtWRKY70','AT3G56400','III'),
]
df = pd.DataFrame(data, columns=['AtWRKY','Locus','Grupo'])
print(f'n = {len(df)}  (AtWRKY52/At5g45270 ausente del proteoma TAIR10)')
print(df.to_string(index=False))

n = 71  (AtWRKY52/At5g45270 ausente del proteoma TAIR10)
 AtWRKY      Locus Grupo
 AtWRKY1  AT2G04880     I
     ...        ...   ...
AtWRKY70  AT3G56400   III


### Tabla S2 — Homólogos de *O. sativa* (39) y *S. lycopersicum* (40) identificados por BLAST

In [1]:
import pandas as pd

WRKY_NAME = {
  'AT2G04880':'AtWRKY1','AT5G56270':'AtWRKY2','AT2G03340':'AtWRKY3',
  'AT1G13960':'AtWRKY4','AT1G55600':'AtWRKY10','AT4G12020':'AtWRKY19',
  'AT4G26640':'AtWRKY20','AT2G30250':'AtWRKY25','AT5G07100':'AtWRKY26',
  'AT4G30935':'AtWRKY32','AT2G38470':'AtWRKY33','AT4G26440':'AtWRKY34',
  'AT2G37260':'AtWRKY44','AT3G01970':'AtWRKY45','AT3G01080':'AtWRKY58',
  'AT4G31800':'AtWRKY18','AT1G80840':'AtWRKY40','AT2G25000':'AtWRKY60',
  'AT1G62300':'AtWRKY6','AT1G68150':'AtWRKY9','AT4G22070':'AtWRKY31',
  'AT1G69810':'AtWRKY36','AT4G04450':'AtWRKY42','AT4G01720':'AtWRKY47',
  'AT1G18860':'AtWRKY61','AT5G15130':'AtWRKY72',
  'AT5G46350':'AtWRKY8','AT2G44745':'AtWRKY12','AT4G39410':'AtWRKY13',
  'AT2G47260':'AtWRKY23','AT5G41570':'AtWRKY24','AT4G18170':'AtWRKY28',
  'AT2G46130':'AtWRKY43','AT5G49520':'AtWRKY48','AT5G43290':'AtWRKY49',
  'AT5G26170':'AtWRKY50','AT5G64810':'AtWRKY51','AT1G64000':'AtWRKY56',
  'AT1G69310':'AtWRKY57','AT2G21900':'AtWRKY59','AT3G62340':'AtWRKY68',
  'AT1G29860':'AtWRKY71','AT5G13080':'AtWRKY75',
  'AT4G24240':'AtWRKY7','AT4G31550':'AtWRKY11','AT2G23320':'AtWRKY15',
  'AT2G24570':'AtWRKY17','AT2G30590':'AtWRKY21','AT3G04670':'AtWRKY39',
  'AT5G28650':'AtWRKY74',
  'AT1G30650':'AtWRKY14','AT5G45050':'AtWRKY16','AT4G01250':'AtWRKY22',
  'AT5G52830':'AtWRKY27','AT4G23550':'AtWRKY29','AT2G34830':'AtWRKY35',
  'AT1G29280':'AtWRKY65','AT3G58710':'AtWRKY69',
  'AT5G24110':'AtWRKY30','AT5G22570':'AtWRKY38','AT4G11070':'AtWRKY41',
  'AT2G46400':'AtWRKY46','AT4G23810':'AtWRKY53','AT2G40750':'AtWRKY54',
  'AT2G40740':'AtWRKY55','AT5G01900':'AtWRKY62','AT1G66600':'AtWRKY63',
  'AT1G66560':'AtWRKY64','AT1G80590':'AtWRKY66','AT1G66550':'AtWRKY67',
  'AT3G56400':'AtWRKY70',
}
WRKY_GROUP = {
  'AT2G04880':'I','AT5G56270':'I','AT2G03340':'I','AT1G13960':'I','AT1G55600':'I',
  'AT4G12020':'I','AT4G26640':'I','AT2G30250':'I','AT5G07100':'I','AT4G30935':'I',
  'AT2G38470':'I','AT4G26440':'I','AT2G37260':'I','AT3G01970':'I','AT3G01080':'I',
  'AT4G31800':'IIa','AT1G80840':'IIa','AT2G25000':'IIa',
  'AT1G62300':'IIb','AT1G68150':'IIb','AT4G22070':'IIb','AT1G69810':'IIb',
  'AT4G04450':'IIb','AT4G01720':'IIb','AT1G18860':'IIb','AT5G15130':'IIb',
  'AT5G46350':'IIc','AT2G44745':'IIc','AT4G39410':'IIc','AT2G47260':'IIc',
  'AT5G41570':'IIc','AT4G18170':'IIc','AT2G46130':'IIc','AT5G49520':'IIc',
  'AT5G43290':'IIc','AT5G26170':'IIc','AT5G64810':'IIc','AT1G64000':'IIc',
  'AT1G69310':'IIc','AT2G21900':'IIc','AT3G62340':'IIc','AT1G29860':'IIc','AT5G13080':'IIc',
  'AT4G24240':'IId','AT4G31550':'IId','AT2G23320':'IId','AT2G24570':'IId',
  'AT2G30590':'IId','AT3G04670':'IId','AT5G28650':'IId',
  'AT1G30650':'IIe','AT5G45050':'IIe','AT4G01250':'IIe','AT5G52830':'IIe',
  'AT4G23550':'IIe','AT2G34830':'IIe','AT1G29280':'IIe','AT3G58710':'IIe',
  'AT5G24110':'III','AT5G22570':'III','AT4G11070':'III','AT2G46400':'III',
  'AT4G23810':'III','AT2G40750':'III','AT2G40740':'III','AT5G01900':'III',
  'AT1G66600':'III','AT1G66560':'III','AT1G80590':'III','AT1G66550':'III','AT3G56400':'III',
}

cols = ['qseqid','sseqid','pident','length','evalue','bitscore']
results = []
for sp, fname in [('Os','03_blast/AT_WRKY_vs_Osativa.tsv'),
                  ('Sl','03_blast/AT_WRKY_vs_Slycopersicum.tsv')]:
    df = pd.read_csv(fname, sep='\t', header=None, names=cols)
    best = df.sort_values('bitscore',ascending=False).drop_duplicates('qseqid').copy()
    best['locus'] = best['qseqid'].str.extract(r'(AT\dG\d+)',expand=False).str.upper()
    best['AtWRKY'] = best['locus'].map(WRKY_NAME)
    best['Grupo'] = best['locus'].map(WRKY_GROUP)
    best['Homólogo'] = best['sseqid'].str.extract(r'^([^|\s]+)',expand=False)
    best['Especie'] = sp
    best['e-value'] = best['evalue'].apply(lambda x: f'{x:.2e}')
    results.append(best[['Especie','AtWRKY','locus','Grupo','Homólogo','pident','length','e-value','bitscore']])

tbl = pd.concat(results).sort_values(['Especie','Grupo','AtWRKY'])
tbl.columns = ['Sp','AtWRKY','Locus At','Grupo','ID homólogo','%id','len_aln','e-value','bitscore']
print(tbl.to_string(index=False))

  Sp    AtWRKY    Locus At Grupo            ID homólogo   %id  len_aln      e-value  bitscore
  Os  AtWRKY1   AT2G04880     I   LOC_Os07g39480.1  44.4      275    7.80e-66     223.0
  Os  AtWRKY2   AT5G56270     I   LOC_Os08g38990.3  43.0      597  8.79e-125     385.0
  ...      ...         ...   ...                ...   ...      ...          ...       ...
  Sl  AtWRKY70  AT3G56400   III                 sp  41.2      187    7.67e-39     137.0


### Nota S1 — Advertencias de IQ-TREE

**Gaps:** `WARNING: 149 sequences contain more than 50% gaps/ambiguity` — esperado con secuencias completas de tres linajes divergentes. No invalida el análisis.

**Composición:** `128 sequences failed composition chi2 test (p-value<5%; df=19)` — indica heterogeneidad composicional entre linajes. IQ-TREE procede normalmente; modelos con composición variable (C60+LG) podrían mejorar el ajuste a mayor costo computacional.

**Renombramiento:** IQ-TREE reemplazó `:` por `_` en 7 headers de *At* con PACid (e.g., `AT_AT2G37260.1|PACid:19638141` → `AT_AT2G37260.1|PACid_19638141`). Los loci `ATxGxxxxx` no se ven afectados.

---
## Referencias

1. Eulgem, T., *et al.* (2000). The WRKY superfamily of plant transcription factors. *Trends in Plant Science* **5**, 199–206. https://doi.org/10.1016/S1360-1385(00)01600-9

2. Rushton, P.J., *et al.* (2010). WRKY transcription factors. *Trends in Plant Science* **15**, 247–258. https://doi.org/10.1016/j.tplants.2010.02.006

3. Rinerson, C.I., *et al.* (2015). The evolution of WRKY transcription factors. *BMC Plant Biology* **15**, 66. https://doi.org/10.1186/s12870-015-0456-y

4. Katoh, K. & Standley, D.M. (2013). MAFFT version 7. *Mol. Biol. Evol.* **30**, 772–780. https://doi.org/10.1093/molbev/mst010

5. Minh, B.Q., *et al.* (2020). IQ-TREE 2. *Mol. Biol. Evol.* **37**, 1530–1534. https://doi.org/10.1093/molbev/msaa015

6. Letunic, I. & Bork, P. (2024). iTOL v6. *Nucleic Acids Res.* **52**, W78–W82. https://doi.org/10.1093/nar/gkae268

7. Goodstein, D.M., *et al.* (2012). Phytozome. *Nucleic Acids Res.* **40**, D1178–D1186. https://doi.org/10.1093/nar/gkr944

8. Altschul, S.F., *et al.* (1990). BLAST. *J. Mol. Biol.* **215**, 403–410. https://doi.org/10.1016/S0022-2836(05)80360-2

---
*Apple MacBook Air · M4 · 24 GB · macOS Tahoe 26.3.1 · scorrea@Sebastians-MacBook-Air*  
*Árbol interactivo:* https://itol.embl.de/tree/201233274241411772770440